import simvoice


In [24]:
from sympy import *

## Lossless Tube Junction


In [25]:
Z1, Z2, F1, B2, D = symbols("Z_1,Z_2,F_1,B_2,D")

M = Matrix([[1, -1], [1 / Z2, 1 / Z1]])
M2 = Matrix([[1, -1], [1 / Z1, 1 / Z2]])

F = M**-1 * M2 * Matrix([[F1], [B2]])  # - Matrix([[F1], [B2]])
F2 = collect(F[0] - F1, (F1, B2))
B1 = collect(F[1] - B2, (F1, B2))
display(F2)
display(B1)

B_2*(Z_1/(Z_1 + Z_2) - Z_2/(Z_1 + Z_2)) + F_1*(2*Z_2/(Z_1 + Z_2) - 1)

B_2*(2*Z_1/(Z_1 + Z_2) - 1) + F_1*(-Z_1/(Z_1 + Z_2) + Z_2/(Z_1 + Z_2))

$$
\begin{aligned}
\begin{bmatrix}F_2\\B_1\end{bmatrix}
&= \begin{bmatrix}F_1\\B_2\end{bmatrix}
+ \frac{Z_1-Z_2}{Z_1+Z_2}\begin{bmatrix}-F_1\\B_2\end{bmatrix}
\end{aligned}
$$
where
$$
Z_k = \frac{\rho c}{A_k}
$$
with the crosssectional area $A_k$ of the $k-th$ tube section, air density $\rho$,
and speed of sound $c$.

# Tube Junction with Yielding Wall


<img src="story95-fig2-4.png" width="50%">

- $U_1$ - input volume flow from the previous junction
- $U_2$ - output volume flow to the next junction
- $U_w$ - wall volume flow
- $R_w$ - wall resistance (simulates the viscosity of the tissue/energy dissipation)
  $$
  R_w = \frac{B}{2l\sqrt{\pi A}}
  $$
- $L_w$ - wall inertance (mass-like qualities of the flesh)
  $$
  L_w = \frac{M}{2l\sqrt{\pi A}}
  $$
- $C_w$ - wall compliance (simulates pressure storage capability)
  $$
  C_w = \frac{B}{2l\sqrt\pi A}
  $$
  Here, $M=1.5$ g/cm^2, $B=1060$ dyne s/cm^3, $K = 33000$ dyne/cm^3


Governing equations. Flow conservation:

$$
U_2 = U_1 + U_w
$$

Pressure conservation:

$$
P_l = P - P_r - P_c,
$$

where $P$ is the sectional pressure, and $P_l$, $P_r$, and $P_c$ are the pressure drops across each lumped element. The equivalent circuit is then given by:

<img src=story95-fig2-5.png width="50%">

$$
\begin{aligned}
U_w(s) &= \frac{1}{sL_w} P_l(s)\\
       &= \frac{1}{sL_w} \left[P(s) - P_r(s) - P_c(s)\right]\\
       &= \frac{1}{sL_w} \left[P(s) - R U_w(s) - \frac{1}{sC_w} U_w(s)\right]\\
\left[1 + \frac{R_v}{sL_w} + \frac{1}{s^2 C_w L_w}\right]U_w(s) &= \frac{1}{sL_w} P(s)\\
\end{aligned}
$$


In [26]:
from sympy.physics.control import TransferFunction

Cw, Lw, Rw, s = symbols("C_w,L_w,R_w,s")

H = TransferFunction(
    *simplify(
        1 / (s * Lw) / (1 + Rw / (s * Lw) + 1 / (s * Cw) / (s * Lw))
    ).as_numer_denom(),
    s,
)
display(H)

TransferFunction(C_w*s, C_w*L_w*s**2 + C_w*R_w*s + 1, s)

The transfer function $H_w(s)$ is then

$$
H_w(s) \triangleq \frac{U_w(s)}{P(s)} = \frac{C_w s}{L_w C_w s^2 + R C_w s + 1 }
$$


Adding the flow yielding to the wall to the flow conservation equation, we now have a modified system of equations:

$$
\begin{aligned}
F_1 + B_1 &= F_2 + B_2\\
\frac{F_1-B_1}{Z_1} &= \frac{F_2-B_2}{Z_2} + U_w
\end{aligned}
$$


Expressing it in a matrix-vector format yields

$$
\begin{aligned}
\begin{bmatrix}
1 & -1\\
1/Z_2 & 1/Z_1
\end{bmatrix}
\begin{bmatrix}
F_2 \\ B_1
\end{bmatrix}
&=
\begin{bmatrix}
1 & -1 & 0\\
1/Z_1 & 1/Z_2 & 1 \\
\end{bmatrix}
\begin{bmatrix}
F_1 \\ B_2 \\ U_w
\end{bmatrix}\\
U_w &= H_w(F_1+B_1)
\end{aligned}
$$


Solving for $\begin{bmatrix}F_1&B_1\end{bmatrix}^T$...

In [27]:
Z1, Z2, F1, B2, Uw = symbols("Z_1,Z_2,F_1,B_2,U_w")

M = Matrix([[1, -1], [1 / Z2, 1 / Z1]])
M2 = Matrix([[1, -1, 0], [1 / Z1, 1 / Z2, 1]])

F = M**-1 * M2 * Matrix([[F1], [B2], [Uw]])  # - Matrix([[F1], [B2]])
F2 = collect(F[0] - F1, (F1, B2))
B1 = collect(F[1] - B2, (F1, B2))
display(F2)
display(B1)

B_2*(Z_1/(Z_1 + Z_2) - Z_2/(Z_1 + Z_2)) + F_1*(2*Z_2/(Z_1 + Z_2) - 1) + U_w*Z_1*Z_2/(Z_1 + Z_2)

B_2*(2*Z_1/(Z_1 + Z_2) - 1) + F_1*(-Z_1/(Z_1 + Z_2) + Z_2/(Z_1 + Z_2)) + U_w*Z_1*Z_2/(Z_1 + Z_2)

$$
\begin{bmatrix}F_2\\B_1\end{bmatrix} =
\begin{bmatrix}F_1\\B_2\end{bmatrix}
+ \frac{Z_1-Z_2}{Z_1+Z_2}
\begin{bmatrix}-F_1 \\ B_2\end{bmatrix}
+ \frac{Z_1Z_2}{Z_1+Z_2} U_w
$$


Ignoring the interaction between $F_1$, $B_2$, and $U_w$, we can observe that the $U_w$ contributes to both $F_1$ and $B_2$ by the same amount. We'll look into the interaction later.

# Tube Junction with Both Yielding Wall and Viscous Loss

Now, we add the first-order viscous loss in series.

<img src=story95-fig2-6.png width = "50%">

Here,

- viscous resistance
  $$
  R_{vsc} = \frac{S}{A^2} \left( \frac{\omega \mu \rho}{2} \right)^\frac{1}{2} l
  $$
- viscous inertance
  $$
  L_{vsc} = \frac{S}{A^2} \left( \frac{\mu \rho}{2 \omega} \right)^\frac{1}{2} l
  $$
- laminar resistance
  $$
  R_{lam} = \frac{8\pi\mu}{A^2}l
  $$
  with $S = 2\sqrt{A\pi}$ is the circumference of a tube section and $\omega$ is
  fixed as $6280$ rad/s. Let $R_v \triangleq R_{vsc} + R_{lam}$.


Define the viscous loss subsystem transfer function

$$
\begin{aligned}
P_v(s) &= U_1(s) (R_v + sL_v)\\
G_v(s) &\triangleq \frac{P_v(s)}{U_2(s)} = R_v + sL_v
\end{aligned}
$$


In [28]:
Rv, Lv = symbols("R_v,L_v")
G = TransferFunction(Rv + s * Lv, 1, s)

Then, the governing equations becomes:
$$
\begin{aligned}
U_1 &= U_2 + U_w\\
P_1 &= P_2 + P_v\\
\end{aligned}
$$
Expanding the flow and total pressure with the partial pressure yields
$$
\begin{aligned}
F_1 + B_1 &= F_2 + B_2 + P_v\\
\frac{F_1-B_1}{Z_1} &= \frac{F_2-B_2}{Z_2} + U_w
\end{aligned}
$$
Note that we're including the viscous loss of Section 2 instead of that of Section 1 as shown in Fig. 2-6. The intention here is to keep the resulting update equation to be responsible only for the junction and propagation through Section 2. It could easily be rederived as shown in Fig. 2-6.

Express in a matrix-vector format:
$$
\begin{equation}
\begin{aligned}
\begin{bmatrix}
1 & -1\\
1/Z_2 & 1/Z_1
\end{bmatrix}
\begin{bmatrix}
F_2 \\ B_1
\end{bmatrix}
&=
\begin{bmatrix}
1 & -1 & 0 & 1\\
1/Z_1 & 1/Z_2 & 1  & 0\\
\end{bmatrix}
\begin{bmatrix}
F_1 \\ B_2 \\ U_w \\ P_v
\end{bmatrix}\\
U_w &= H_w(F_2+B_2) \\
P_v &= G_v(F_2-B_2)/Z_2
\end{aligned}
\end{equation}
$$
Solving for $F_2$ and $B_1$...

In [29]:
Z1, Z2, F1, B2, Uw, Pv = symbols("Z_1,Z_2,F_1,B_2,U_w,P_v")

M = Matrix([[1, -1], [1 / Z2, 1 / Z1]])
M2 = Matrix([[1, -1, 0, 1], [1 / Z1, 1 / Z2, 1, 0]])

F = M**-1 * M2 * Matrix([[F1], [B2], [Uw], [Pv]])  # - Matrix([[F1], [B2]])
F2 = collect(F[0] - F1, (F1, B2, Uw, Pv))
B1 = collect(F[1] - B2, (F1, B2, Uw, Pv))
display(F2)
display(B1)

B_2*(Z_1/(Z_1 + Z_2) - Z_2/(Z_1 + Z_2)) + F_1*(2*Z_2/(Z_1 + Z_2) - 1) + P_v*Z_2/(Z_1 + Z_2) + U_w*Z_1*Z_2/(Z_1 + Z_2)

B_2*(2*Z_1/(Z_1 + Z_2) - 1) + F_1*(-Z_1/(Z_1 + Z_2) + Z_2/(Z_1 + Z_2)) - P_v*Z_1/(Z_1 + Z_2) + U_w*Z_1*Z_2/(Z_1 + Z_2)

Final expression:

$$
\begin{bmatrix}F_2\\B_1\end{bmatrix} =
\begin{bmatrix}F_1\\B_2\end{bmatrix}
+ \frac{Z_1-Z_2}{Z_1+Z_2}
\begin{bmatrix}-F_1 \\ B_2\end{bmatrix}
+ \frac{Z_1Z_2}{Z_1+Z_2}U_w
+ \frac{1}{Z_1+Z_2} \begin{bmatrix}Z_2\\-Z_1\end{bmatrix} P_v
$$


Like $U_w$, $P_v$ term simply adds to the previous result. Again, the interactions have not been incorporated yet.

# Bilinear transformation of $H_w(s)$ and $G_v(s)$


To model how $U_w$ and $P_v$ interact with the other parts of the network, we first take the model from the continueous-time domain to the discrete-time domain. The transfer functions $H_w(s)$ and $G_v(s)$ are discretized by the bilinear transformation.

We have

$$
\begin{aligned}
H_w(s) &= \frac{C_w s}{L_w C_w s^2 - R C_w s -1 }\\
G_v(s) &= R_l + R_v + sL_v
\end{aligned}
$$


Bilinear transformation converts continuous-time TF to discrete-time by substituting

$$
s = \frac{2}{T}\frac{1-z^{-1}}{1+z^{-1}}
$$


In [30]:
zi = symbols("z^{-1}")

print("H(s)")
display(H)

# Define symbols
T = symbols("T", positive=True, real=True)
s2zi = 2 / T * (1 - zi) / (1 + zi)

Hz = TransferFunction(
    *simplify(H.to_expr().subs(s, s2zi)).as_numer_denom(), zi
).expand()
Hz_num = Hz.num.as_poly(zi)
Hz_den = Hz.den.as_poly(zi)
print("H(z)")
display(TransferFunction(Hz_num.as_expr(), Hz_den.as_expr(), zi))
# a2, a1, a0 = Hz_den.coeffs()
# b2, b0 = Hz_num.coeffs()
# print("Hz coefficients (b then a)")
# display(simplify(Matrix([b0, 0, b2]).T/T**2))
# display(simplify(Matrix([a0, a1, a2]).T/T**2))


Gz = TransferFunction(
    *simplify(G.to_expr().subs(s, s2zi)).as_numer_denom(), zi
).expand()
Gz_num = Gz.num.as_poly(zi)
Gz_den = Gz.den.as_poly(zi)
print("G(z)")
display(TransferFunction(Gz_num.as_expr(), Gz_den.as_expr(), zi))
# a1, a0 = Gz_den.coeffs()
# b1, b0 = Gz_num.coeffs()
# print("Gz coefficients (b then a)")
# display(simplify(Matrix([b0, b1]).T/T))
# display(Matrix([a0, a1]).T/T)

H(s)


TransferFunction(C_w*s, C_w*L_w*s**2 + C_w*R_w*s + 1, s)

H(z)


TransferFunction(-2*C_w*T*z^{-1}**2 + 2*C_w*T, 4*C_w*L_w + 2*C_w*R_w*T + T**2 + z^{-1}**2*(4*C_w*L_w - 2*C_w*R_w*T + T**2) + z^{-1}*(-8*C_w*L_w + 2*T**2), z^{-1})

G(z)


TransferFunction(2*L_v + R_v*T + z^{-1}*(-2*L_v + R_v*T), T*z^{-1} + T, z^{-1})

The DT version of $H_w(s)$ producing the flow into the yielding wall $U_w$ from
total pressure $P$ is then defined as

$$
\begin{aligned}
H(z) &= \frac{b_0 + b_2 z^{-2}}{1 - a_1 z^{-1} - a_2 z^{-2}}\\
\text{where}\\
b_0 &= \frac{2\tilde{C}_w}{a_0}\\
b_2 &= \frac{-2\tilde{C}_w}{a_0} = -b_0\\
a_1 &= \frac{8\tilde{C}_w\tilde{L}_w - 2}{a_0}\\
a_2 &= \frac{2\tilde{C}_wR_w-4\tilde{C}_w\tilde{L}_w-1}{a_0}\\
\text{with}\\
a_0 &= (4\tilde{C}_w\tilde{L}_w+2\tilde{C}_wR_w+1)\\
\tilde{C}_w &= C_w/T\\
\tilde{L}_w &= L_w/T\\
\end{aligned}
$$


Likewise, the DT system for the viscous loss is given by

$$
\begin{aligned}
G(z) &= \frac{\beta_0 + \beta_1 z^{-1}}{1 - z^{-1}}\\
\text{where}\\
\beta_0 &= 2\tilde{L}_v R_v\\
\beta_1 &= R_v - 2\tilde{L}_v\\
\text{with}\\
\tilde{L}_v &= L_v/T\\
\end{aligned}
$$


Recall that the $H(s)$ and $G(s)$ defines the relationships:

$$
\begin{aligned}
U_w &= H(F_2+B_2)\\
P_v &= G(F_2-B_2)/Z_2\\
\end{aligned}
$$


They can now be written as the following difference equations:
$$
\begin{aligned}
u_{w,n} + a_1 u_{w,n-1} + a_2 u_{w,n-2} &= b_0(f_{2,n}+b_{2,n}) + b_2(f_{2,n-2}+b_{2,n-2})\\
p_{v,n} + p_{v,n-1} &= \tilde{\beta}_0 (f_{2,n}-b_{2,n}) + \tilde{\beta}_1 (f_{2,n-1}-b_{2,n-1})\\
\text{where}\\
\tilde{\beta}_0 &\triangleq \beta_0 / Z_2\\
\tilde{\beta}_1 &\triangleq \beta_1 / Z_2\\
\end{aligned}
$$

The current values of $F_2$ and $B_2$, $f_{2,n}$ and $b_{2,n}$, respectively, are also parts of the overall equation to be solved in Eq. (1). To incorporate the effects of $H_w(z)$ and $G_v(z)$, the difference equations are regrouped so that all the old values are bunched into a signal:


$$
\begin{aligned}
u_{w,n} &= b_0(f_{2,n}+b_{2,n}) + \tilde{u}_{w,n}\\
p_{v,n} &= \tilde{\beta}_0 (f_{2,n}-b_{2,n}) + \tilde{p}_{v,n}\\
\text{where}\\
\tilde{u}_{w,n} &\triangleq b_2(f_{2,n-2}+b_{2,n-2}) - a_1 u_{w,n-1} - a_2 u_{w,n-2}  \\
\tilde{p}_{v,n} &\triangleq \tilde{\beta}_1 (f_{2,n-1} - b_{2,n-1}) - p_{v,n-1}\\
\end{aligned}
$$

Note: $b_0$ roughly corresponds to $\alpha[n]$ and $\tilde{u}_{w,n}$ to $\beta[n]$ of Eq.
(2.46) in (Story, 1995). On the other hand, $P_v$ was not split into
current and past parts in (Story, 1995).

Denoting $\tilde{u}_{w,n}$ as $\tilde{U}_w$ and $\tilde{p}_{v,n}$ as $\tilde{P}_v$ and substituting them into Eq. (1) yield


$$
\tag{2}
\begin{aligned}
\begin{bmatrix}
1 & -1\\
1/Z_2 & 1/Z_1
\end{bmatrix}
\begin{bmatrix}
F_2 \\ B_1
\end{bmatrix}
&=
\begin{bmatrix}
1 & -1 & 0 & 1\\
1/Z_1 & 1/Z_2 & 1  & 0\\
\end{bmatrix}
\begin{bmatrix}
F_1 \\ B_2 \\ b_0(F_2+B_2) + \tilde{U}_w \\
\tilde{\beta}_0(F_2-B_2) + \tilde{P}_v \\
\end{bmatrix}\\
\begin{bmatrix}
1-\tilde\beta_0 & -1\\
1/Z_2-b_0 & 1/Z_1
\end{bmatrix}
\begin{bmatrix}
F_2 \\ B_1
\end{bmatrix}
&=
\begin{bmatrix}
1 & -1-\tilde{\beta}_0 & 0 & 1\\
1/Z_1 & 1/Z_2+b_0 & 1  & 0\\
\end{bmatrix}
\begin{bmatrix}
F_1 \\ B_2 \\
\tilde{U}_w \\
\tilde{P}_v \\
\end{bmatrix}\\
\end{aligned}
$$


The outputs ($F_2$ and $B_1$) no longer are part of the right hand side. 

Solving this equation...

In [31]:
# Z1, Z2, F1, B2, Uw, Pv = symbols("Z_1,Z_2,F_1,B_2,U_w,P_v")
b0, beta0 = symbols(r"b_0,\beta_0")

M = Matrix([[1 - beta0 / Z2, -1], [1 / Z2 - b0, 1 / Z1]])
M2 = Matrix([[1, -1 - beta0 / Z2, 0, 1], [1 / Z1, 1 / Z2 + b0, 1, 0]])
display(M)
display(M2)
F_ = M**-1 * M2 * Matrix([[F1], [B2], [Uw], [Pv]])

d = symbols("D")
dexpr = Z1 + Z2 - Z1 * Z2 * b0 - beta0
print("D")
display(dexpr)
F = F_.subs(dexpr, d)

F2 = collect(F[0, 0] - F1, (F1, B2, Uw, Pv), evaluate=False)
B1 = collect(F[1, 0] - B2, (F1, B2, Uw, Pv), evaluate=False)
# (1 + b0 * Z1 * Z2 / d) *
print("F2(F1,B2,Uw,Pv)")
display(together(F2[F1].subs(d, dexpr)).as_numer_denom()[0].expand())
display(together(F2[B2].subs(d, dexpr)).as_numer_denom()[0].expand())
display(together(F2[Uw].subs(d, dexpr)).as_numer_denom()[0].expand())
display(together(F2[Pv].subs(d, dexpr)).as_numer_denom()[0].expand())
print("B1(F1,B2,Uw,Pv)")
display(together(B1[F1].subs(d, dexpr)).as_numer_denom()[0].expand())
display(together(simplify(B1[B2]).subs(d, dexpr)).as_numer_denom()[0].expand())
display(together(B1[Uw].subs(d, dexpr)).as_numer_denom()[0].expand())
display(together(B1[Pv].subs(d, dexpr)).as_numer_denom()[0].expand())


Matrix([
[1 - \beta_0/Z_2,    -1],
[   -b_0 + 1/Z_2, 1/Z_1]])

Matrix([
[    1, -1 - \beta_0/Z_2, 0, 1],
[1/Z_1,      b_0 + 1/Z_2, 1, 0]])

D


-Z_1*Z_2*b_0 + Z_1 + Z_2 - \beta_0

F2(F1,B2,Uw,Pv)


...

B1(F1,B2,Uw,Pv)


Z_1*Z_2*b_0 - Z_1 + Z_2 - \beta_0

Z_1*Z_2*b_0 - 2*Z_1*\beta_0*b_0 + Z_1 - Z_2 + \beta_0

Z_1*Z_2 - Z_1*\beta_0

Z_1*Z_2*b_0 - Z_1

we get:

$$
\begin{aligned}
\begin{bmatrix}
F_2 \\ B_1
\end{bmatrix}
&=
\left(1 + \frac{b_0 Z_1 Z_2}{D}\right)
\begin{bmatrix}
F_1 \\ B_2 \\
\end{bmatrix}
+
\frac{1}{D}
\begin{bmatrix}
-Z_1 + Z_2 + \beta_0 & Z_1 - Z_2 - \beta_0 & Z_1Z_2 & Z_2\\
 Z_1 - Z_2 - \beta_0 & Z_1 - Z_2 + \beta_0 - 2b_0\beta_0Z_1 & Z_1Z_2-\beta_0Z_1 & Z_1Z_2b_0 - Z_1\\
\end{bmatrix}
\begin{bmatrix}
F_1 \\ B_2 \\
\tilde{U}_w \\
\tilde{P}_v \\
\end{bmatrix}\\
&=
\frac{Z_1 + Z_2 - \beta_0}{D}
\begin{bmatrix}
F_1 \\ B_2 \\
\end{bmatrix}
+
\frac{Z_1 - Z_2}{D}
\begin{bmatrix}
-F_1 \\ B_2 \\
\end{bmatrix}
+
\frac{\beta_0}{D}
\begin{bmatrix}
 1 & -1\\
-1 & 1 - 2b_0 Z_1\\
\end{bmatrix}
\begin{bmatrix}
F_1 \\ B_2 \\
\end{bmatrix}\\
&\quad+
\frac{Z_1Z_2}{D}
\begin{bmatrix}
             1\\
1-\beta_0/Z_2\\
\end{bmatrix}
\tilde{U}_w 
+
\frac{1}{D}
\begin{bmatrix}
             Z_2\\
-Z_1 + b_0 Z_1Z_2\\
\end{bmatrix}
\tilde{P}_v
\end{aligned}
$$

where,

$$
D \triangleq Z_1 + Z_2 - b_0 Z_1 Z_2 - \beta_0
$$


While it has a reminiscence of the original lattice structure (the first two terms of the right-hand side) the third term and the modification to the first term make the lattice structure no longer provides clear computational benefit which was obvious for the lossless tubes. Although there may be a numerical benefit to use an alternate form, the single matrix multiplication form will be used:

$$
\begin{aligned}
\begin{bmatrix}
F_2 \\ B_1
\end{bmatrix}
&=
\mathbf{M}
\begin{bmatrix}
F_1 \\ B_2 \\
\tilde{U}_w \\
\tilde{P}_v \\
\end{bmatrix}
\end{aligned}
$$

where

$$
\mathbf{M} \triangleq
\frac{1}{D}
\begin{bmatrix}
2 Z_2 & Z_1 - Z_2 - \beta_0 & Z_1Z_2 & Z_2\\
 Z_1 - Z_2 - \beta_0 & 2 Z_1(1 - b_0\beta_0) & Z_1(Z_2-\beta_0) & Z_1(Z_2b_0 - 1)\\
\end{bmatrix}
$$


Converting the impedances to the cross-sectional areas:

$$
\mathbf{M} \triangleq
\frac{1}{D}\frac{\rho c}{A_1 A_2}
\begin{bmatrix}
2 A_1 & A_2 - A_1 - \tilde{\beta}_0 A_1 A_2 & \rho c & A_1\\
A_2 - A_1 - \tilde{\beta}_0 A_1 A_2 & 2 A_2 (1 - \tilde{b}_0\tilde{\beta}_0) & \rho c(1-\tilde{\beta}_0 A_2) & \tilde{b}_0 - A_2\\
\end{bmatrix}
$$
where
$$\begin{aligned}
\tilde{b}_0 &\triangleq \rho c b_0\\
\tilde{\beta}_0 &\triangleq \frac{\beta_0}{\rho c}\\
\tilde{D} &\triangleq \frac{\rho c}{A_1A_2} D = A_2 + A_1 - \tilde{b}_0 -  \tilde{\beta_0}A_1A_2\\
\end{aligned}$$


# Implementation


- Input: $f_{1,n}$, $b_{2,n}$
- Output: $f_{2,n+1}$, $b_{1,n+1}$
- States (internal to each junction): $u_{w,n-1}$, $u_{w,n-2}$, $p_{2,n-1}$, $p_{2,n-2}$, $p_{v,n-1}$, $u_{2,n-1}$


For time step $n$,

1. Calculate the contributions from the past flow and total pressure:
   $$
   \begin{aligned}
   \tilde{u}_{w} &:= b_2 p_{2,n-2} - a_1 u_{w,n-1} - a_2 u_{w,n-2}  \\
   \tilde{p}_{v} &:= \tilde{\beta}_1 u_{2,n-1} - p_{v,n-1}\\
   \end{aligned}
   $$
2. Calculate the current outputs
   $$
   \begin{bmatrix}
   f_{2,n} \\ b_{1,n}
   \end{bmatrix}
   :=
   \mathbf{M}
   \begin{bmatrix}
   f_{1,n} \\ b_{2,n} \\
   \tilde{u}_{w,n} \\
   \tilde{p}_{v,n} \\
   \end{bmatrix}
   $$
3. Update the second tube's flow and total pressure states
   $$
   \begin{aligned}
   p_{2,n-2} &:= p_{2,n-1}\\
   p_{2,n-1} &:= f_{2,n} + b_{2,n}\\
   u_{2,n-1} &:= (f_{2,n} - b_{2,n})/Z_2\\
   \end{aligned}
   $$
4. Update the yielding wall flow and the viscous pressure loss
   $$
   \begin{aligned}
   u_{w,n-2} &:= u_{w,n-1}\\
   u_{w,n-1} &:= b_0 p_{2,n-1} + \tilde{u}_w\\
   p_{v,n-1} &:= \tilde{\beta}_0 u_{2,n-1} + \tilde{p}_v\\
   \end{aligned}
   $$
5. Increment time step: $n:=n+1$


## Comparison to Story 1995

Aside from the handling of the series (viscous loss) elements, there is a slight 
difference in the derivation of the shunt elements here vs. (Story, 1995). 
The former discretized the entire second-order shunt model $H(s)$ using the bilinear
transformation while the latter used the trapezoid rule on on $1/sL_w$ and $1/sC_w$
blocks in Fig. 2-5.

We can re-derive (Story, 1995) using the bilinear transformation instead of the 
trapezoid rule as they are equivalent (at least?) for the first order model.

Convert each continuous-time block in Fig. 2-5 to discrete-time by the bilinear transformation:

$$\begin{align}
P_l &= P - P_r - P_c\\
\frac{U_w}{P_l} &= \frac{1+z^{-1}}{2\tilde{L}_w(1-z^{-1})}\\
\frac{P_r}{U_w} &= R_w\\
\frac{P_c}{U_w} &= \frac{1+z^{-1}}{2\tilde{C}_w(1-z^{-1})}\\
\end{align}$$

Express the transfer functions as difference equations and introduce $\beta_{l,n-1}$ and $\beta_{c,n-1}$:

$$\begin{aligned}
p_{l,n} &= p_n - p_{r,n} - p_{c,n}\\
u_{w,n} &= \alpha_l p_{l,n} + \alpha_l p_{l,n-1} + u_{w,n-1} = \alpha_l p_{l,n} + \beta_{l,n-1} \quad (\beta_{l,n-1} \triangleq \alpha_l p_{l,n-1} + u_{w,n-1})\\
p_{r,n} &= \alpha_r u_{w,n}\\
p_{c,n} &= \alpha_c u_{w,n} + \alpha_c u_{w,n-1} + p_{c,n-1} = \alpha_c u_{w,n} + \beta_{c,n-1} \quad (\beta_{c,n-1} \triangleq p_{c,n-1} + \alpha_c u_{w,n-1} )\\
\end{aligned}$$

Substituting the other equations into the $u_{w,n}$ equation and subsequent algebraic manipulation yield Eq. (2.46):
$$\begin{aligned}
u_{w,n} &= \alpha_l p_{l,n} + \beta_{l,n-1} \\
&= \alpha_l (p_n - p_{c,n} - p_{r,n}) + \beta_{l,n-1}\\
&= \alpha_l p_n - \alpha_l(\alpha_c u_{w,n} + \beta_{c,n-1}) - \alpha_l p_{r,n} + \beta_{l,n-1}\\
u_{w,n} + \alpha_l \alpha_c u_{w,n} + \alpha_l \alpha_r u_{w,n}&= \alpha_l p_n - \alpha_l \beta_{c,n-1} + \beta_{l,n-1}\\
(1 + \alpha_l \alpha_c + \alpha_l \alpha_r) u_{w,n}&= \alpha_l p_n - \alpha_l \beta_{c,n-1} + \beta_{l,n-1}\\
u_{w,n}&= \frac{\alpha_l}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r} p_n - \frac{\alpha_l \beta_{c,n-1} + \beta_{l,n-1}}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r}\\
&= \alpha p_n + \beta_n
\end{aligned}$$

Taking it a bit further and substituting the definitions of $\beta_{l,n-1}$ and $\beta_{c,n-1}$ yield
$$\begin{aligned}
\beta_n &= -\frac{\alpha_l \beta_{c,n-1} + \beta_{l,n-1}}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r}\\
&=-\frac{\alpha_l (p_{c,n-1} + \alpha_c u_{w,n-1}) + (\alpha_l p_{l,n-1} + u_{w,n-1})}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r}\\
&=-\frac{\alpha_l (p_{c,n-1} + p_{l,n-1}) + (1 + \alpha_l \alpha_c) u_{w,n-1}}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r}\\
&=-\frac{\alpha_l p_{n-1} + (1 + \alpha_l \alpha_c - \alpha_r \alpha_l) u_{w,n-1}}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r}
\end{aligned}$$
Now $\beta_n$ is expressed purely with the input $p_n$ and output $u_{w,n}$, and it is apparent that this discretized model is a first-order model (only ($n-1$) past samples are needed). In contrary $\tilde{u}_n$ above requires the ($n-2$) past samples. Recall:
$$
\tilde{u}_{w,n} = b_2 p_{n-2} - a_1 u_{w,n-1} - a_2 u_{w,n-2}
$$

The coefficients of the present model can also be expressed with $\alpha_l$, $\alpha_c$, and $\alpha_r$:
$$
\begin{aligned}
a_0 &= \alpha_c^{-1}\alpha_l^{-1} + \alpha_c^{-1} \alpha_r + 1\\
b_2 &= -\alpha_c^{-1} / a_0 = -\frac{\alpha_l}{1 + \alpha_l \alpha_r + \alpha_l \alpha_c}
\\
a_1 &= \frac{2\alpha_c^{-1} \alpha_l^{-1} - 2}{\alpha_c^{-1} \alpha_l^{-1} + \alpha_c^{-1} \alpha_r + 1}
= \frac{2 - 2 \alpha_l \alpha_c}{1 + \alpha_l \alpha_r + \alpha_l \alpha_c}\\
a_2 &= \frac{\alpha_c^{-1} \alpha_r - \alpha_c^{-1} \alpha_l^{-1} - 1}{\alpha_c^{-1} \alpha_l^{-1} + \alpha_c^{-1} \alpha_r + 1}
= -\frac{1 + \alpha_l \alpha_c - \alpha_l \alpha_r}{1 + \alpha_l \alpha_r + \alpha_l \alpha_c}\\
\end{aligned}
$$

The first coefficient of the both models are the same, except they are multiplying $p_{n-1}$ and $p_{n-2}$.
The last coefficients are also the same, again with the difference in their multiplicands.
The extra coefficient $a_1$ of the 2nd order model compensates for these differences in the multiplicands.